# `mb.fit` against five libraries on held-out data

`mb.fit(data, "y")` searches for a model specification within a budget of
fits and a time limit, then fits it. It is the call an automated analyst uses
when it has no formula. The held-out comparison on this page measures that call
on the test rows of 95 real tables and on seven generators with known structure,
against CatBoost, LightGBM, XGBoost, EBM (the Explainable Boosting Machine of
InterpretML) and a ridge or logistic regression. A second comparison, at the end
of the page, sets `fit` against its previous version, which used default
learners without a budgeted search. The
[scorecard rubric](overview.ipynb#scorecard-dimensions-at-a-glance)
defines the six measured dimensions and the six left unmeasured.

The comparison matters because the libraries trade off different goals. A
tree ensemble is usually the most accurate predictor; an additive model states
which columns matter and how. The scores below quantify that trade-off on the
same tasks.

**Main findings.**

- In the held-out comparison mboostrs received the highest grades on D1b, D9
  and D10. On the
  planted tasks, the shared permutation rule found 70 of 80 true predictors
  with a 1.4% false-discovery rate. The formula returned to the user had
  64.5% false discoveries under its own scoring. For nominal 90% prediction
  intervals, median held-out coverage was 90%.
- CatBoost scores highest on prediction (D11). mboostrs has a higher test loss
  than CatBoost on 24 of 33 regression tables.
- mboostrs adopts no interaction terms, so it misses a planted pure interaction
  that every tree library finds.
- mboostrs completes 71% of the real-data runs, the lowest robustness score
  (D6) of the six libraries, and takes about 34 seconds per follow-up fit at
  50,000 rows, the second-lowest speed score (D5) after EBM.
- Against its previous version, `fit` has lower held-out loss on 38 of 62
  paired tables, higher on 20 and equal on 4.

For both comparisons the protocol, the tasks and the decision criteria were
fixed, and the file digests recorded, before the first run.

In [1]:
import json
import sys
from pathlib import Path

import polars as pl
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(ROOT))
from docs.book.cases.bench_tables import table

STUDY = ROOT / "docs/benchmarks/competitive_heldout"
analysis = json.loads((STUDY / "analysis.json").read_text())
ARMS = ["mboostrs", "catboost", "lightgbm", "xgboost", "ebm", "reference"]
NAMES = {"mboostrs": "mboostrs", "catboost": "CatBoost", "lightgbm": "LightGBM",
         "xgboost": "XGBoost", "ebm": "EBM", "reference": "Ridge/logistic"}
cards = {a: json.loads((STUDY / "scorecards" / f"{a}.json").read_text()) for a in ARMS}

## Score sheet

Each library was called once per task with its default settings, on one thread.
A score runs from 1 to 5 against the scorecard's anchors, and higher is better.
The six measured dimensions are D1b discovery from data alone, D5 iteration
speed, D6 robustness, D9 prediction-interval calibration, D10 reproducibility,
and D11 predictive validity. The remaining dimensions stay unmeasured. The
scorecard reports an aggregate only for a complete profile. Each library's report gives the measurements behind every score.

In [2]:
dimension_rows = []
for i, d in enumerate(cards["mboostrs"]["dimensions"]):
    row = {"dimension": f"{d['id']} {d['name']}"}
    for a in ARMS:
        entry = cards[a]["dimensions"][i]
        row[NAMES[a]] = (str(entry["score"]).removesuffix(".0") if entry["status"] == "scored"
                         else entry["status"].replace("_", " "))
    dimension_rows.append(row)
display(table(
    pl.DataFrame(dimension_rows), stub="dimension", stubhead="Dimension",
    title="mboostrs-scorecard/2.1, fully automated mode",
    subtitle="Scores 1 to 5, higher is better",
    notes=(f"Scorecard SHA-256 {cards['mboostrs']['scorecard_sha256'][:12]}…; "
           "one report per library in docs/benchmarks/competitive_heldout/scorecards/.",
           "D11 has no score for the reference because the reference is what D11 measures against."),
))

GT(_tbl_data=shape: (12, 7)
┌────────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────────┐
│ dimension      ┆ mboostrs   ┆ CatBoost   ┆ LightGBM   ┆ XGBoost    ┆ EBM        ┆ Ridge/logistic │
│ ---            ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---            │
│ str            ┆ str        ┆ str        ┆ str        ┆ str        ┆ str        ┆ str            │
╞════════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════════════╡
│ D1a            ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured     │
│ Relationship   ┆            ┆            ┆            ┆            ┆            ┆                │
│ recovery unde… ┆            ┆            ┆            ┆            ┆            ┆                │
│ D1b            ┆ 3          ┆ 2          ┆ 2          ┆ 2          ┆ 2          ┆ 1              │
│ Relationship   ┆            ┆            ┆            ┆            ┆            ┆                │
│ discovery fro… ┆            ┆            ┆            ┆            ┆            ┆                │
│ D2 Causal      ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured     │
│ humility and   ┆            ┆            ┆            ┆            ┆            ┆                │
│ support        ┆            ┆            ┆            ┆            ┆            ┆                │
│ D3 Explanation ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured     │
│ quality        ┆            ┆            ┆            ┆            ┆            ┆                │
│ D4 Importance  ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured     │
│ quantification ┆            ┆            ┆            ┆            ┆            ┆                │
│ …              ┆ …          ┆ …          ┆ …          ┆ …          ┆ …          ┆ …              │
│ D7 Human-frien ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured     │
│ dliness        ┆            ┆            ┆            ┆            ┆            ┆                │
│ D8 Agent-frien ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured ┆ unmeasured     │
│ dliness        ┆            ┆            ┆            ┆            ┆            ┆                │
│ D9 Uncertainty ┆ 4          ┆ 2          ┆ 2          ┆ 2          ┆ 1          ┆ 3              │
│ honesty        ┆            ┆            ┆            ┆            ┆            ┆                │
│ (calibr…       ┆            ┆            ┆            ┆            ┆            ┆                │
│ D10 Reproducib ┆ 3          ┆ 2          ┆ 2          ┆ 2          ┆ 2          ┆ 2              │
│ ility and      ┆            ┆            ┆            ┆            ┆            ┆                │
│ audita…        ┆            ┆            ┆            ┆            ┆            ┆                │
│ D11 Predictive ┆ 3          ┆ 4          ┆ 3          ┆ 2.5        ┆ 3.5        ┆ not applicable │
│ validity       ┆            ┆            ┆            ┆            ┆            ┆                │
└────────────────┴────────────┴────────────┴────────────┴────────────┴────────────┴────────────────┘, _body=<great_tables._gt_data.Body object at 0x10b30a330>, _boxhead=Boxhead([ColInfo(var='dimension', type=<ColInfoTypeEnum.stub: 2>, column_label='dimension', column_align='left', column_width=None, stub_level=0), ColInfo(var='mboostrs', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs', column_align='left', column_width=None, stub_level=None), ColInfo(var='CatBoost', type=<ColInfoTypeEnum.default: 1>, column_label='CatBoost', column_align='left', column_width=None, stub_level=None), ColInfo(var='LightGBM', type=<ColInfoTypeEnum.default: 1>, column_label='LightGBM', column_align='left', column_width=None, stub_level=None), ColInfo(var='XGBoost', type=<ColInfoTypeEnum.default: 1>, column_label='XGBoost', column_align='lef

mboostrs has the highest score on discovery (D1b), uncertainty (D9) and
reproducibility (D10). It scores below the tree libraries on speed (D5),
robustness (D6) and, against CatBoost, prediction (D11).

## Tasks

- **Real data.** 95 tables from OpenML, the UCI repository and
  `mboostrs.datasets`: 45 regression and 50 binary. Each runs on three seeds
  with an 80/20 split, at most 4,000 training rows and 2,000 test rows.
- **Planted structure.** Seven generators with known truth, five draws each:
  smooth additive effects (S1), a pure interaction between two columns (S2), an
  effect on the spread only (S3), no effect at all (S4), 200 training rows (S5),
  near-duplicate copies of true predictors (S6), and binary additive (S7).

## Prediction (D11)

For each real task, the gap of a library is how much higher its mean test loss
is than the lowest loss among the libraries that completed the task. Test loss
is mean squared error for regression and log loss for binary targets. The
best complete arm is picked after the test losses are known. D11 grading uses
the prespecified ridge/logistic reference for its baseline comparison.

In [3]:
d11_rows = []
for kind, label in (("real_regression", "Regression (45 tasks)"), ("real_binary", "Binary (50 tasks)")):
    for a in ARMS:
        s = analysis["D11"][kind]["by_arm"][a]
        d11_rows.append({"group": label, "library": NAMES[a], "completed": s["complete_tasks"],
                         "median_gap_pct": 100 * s["median_gap_to_best_complete"],
                         "within_5pct": s["within_5pct_of_best"]})
display(table(
    pl.DataFrame(d11_rows), stub="library", stubhead="Library", group="group",
    title="Gap to the best library on each real task",
    subtitle="Tasks completed on all three seeds; gap in percent of the best loss",
    labels={"completed": "Tasks completed", "median_gap_pct": "Median gap (%)",
            "within_5pct": "Tasks within 5% of best"},
    better={"median_gap_pct": "min"},
    decimals=1,
    notes=("A task a library fails on any seed is excluded from that library's gap.",),
))

GT(_tbl_data=shape: (12, 5)
┌───────────────────────┬────────────────┬───────────┬────────────────┬─────────────┐
│ group                 ┆ library        ┆ completed ┆ median_gap_pct ┆ within_5pct │
│ ---                   ┆ ---            ┆ ---       ┆ ---            ┆ ---         │
│ str                   ┆ str            ┆ i64       ┆ f64            ┆ i64         │
╞═══════════════════════╪════════════════╪═══════════╪════════════════╪═════════════╡
│ Regression (45 tasks) ┆ mboostrs       ┆ 33        ┆ 35.28          ┆ 8           │
│ Regression (45 tasks) ┆ CatBoost       ┆ 45        ┆ 0.0            ┆ 34          │
│ Regression (45 tasks) ┆ LightGBM       ┆ 45        ┆ 9.54           ┆ 15          │
│ Regression (45 tasks) ┆ XGBoost        ┆ 45        ┆ 19.29          ┆ 6           │
│ Regression (45 tasks) ┆ EBM            ┆ 39        ┆ 7.79           ┆ 18          │
│ …                     ┆ …              ┆ …         ┆ …              ┆ …           │
│ Binary (50 tasks)     ┆ CatBoost       ┆ 50        ┆ 3.89           ┆ 29          │
│ Binary (50 tasks)     ┆ LightGBM       ┆ 50        ┆ 29.35          ┆ 6           │
│ Binary (50 tasks)     ┆ XGBoost        ┆ 50        ┆ 35.71          ┆ 4           │
│ Binary (50 tasks)     ┆ EBM            ┆ 47        ┆ 1.93           ┆ 33          │
│ Binary (50 tasks)     ┆ Ridge/logistic ┆ 50        ┆ 20.53          ┆ 14          │
└───────────────────────┴────────────────┴───────────┴────────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x10bb003e0>, _boxhead=Boxhead([ColInfo(var='group', type=<ColInfoTypeEnum.row_group: 3>, column_label='group', column_align='left', column_width=None, stub_level=None), ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None, stub_level=0), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='Tasks completed', column_align='right', column_width=None, stub_level=None), ColInfo(var='median_gap_pct', type=<ColInfoTypeEnum.default: 1>, column_label='Median gap (%)', column_align='right', column_width=None, stub_level=None), ColInfo(var='within_5pct', type=<ColInfoTypeEnum.default: 1>, column_label='Tasks within 5% of best', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10cfacfe0>, _spanners=Spanners([]), _heading=Heading(title='Gap to the best library on each real task', subtitle='Tasks completed on all three seeds; gap in percent of the best loss', preheader=None), _stubhead='Library', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10bd07fe0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d04ba70>, _source_notes=["A task a library fails on any seed is excluded from that library's gap."], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09C0B0>, mask=None), grpname=None, colname='completed', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09C0B0>, mask=None), grpname=None, colname='completed', rownum=6, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09C0B0>, mask=None), grpname=None, colname='median_gap_pct', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09C0B0>, mask=None), grpname=None, colname='median_gap_pct', rownum=6, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09C0B0>, mask=None), grpname=None, colname='within_5pct', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library

In [4]:
reg = analysis["D11"]["real_regression"]["paired"]
binary = analysis["D11"]["real_binary"]["paired"]
cb, ref = reg["mboostrs_vs_catboost"], reg["mboostrs_vs_reference"]
display(Markdown(
    f"On regression, mboostrs has lower loss than the ridge reference on {ref['mboostrs_better']} "
    f"of {ref['n']} tasks (Wilcoxon p = {ref['wilcoxon_p']:.0e}) and higher loss than CatBoost on "
    f"{cb['mboostrs_worse']} of {cb['n']} (median loss ratio {cb['median_ratio']:.2f}, "
    f"p = {cb['wilcoxon_p']:.3f}). On binary tasks it beats the logistic reference on "
    f"{binary['mboostrs_vs_reference']['mboostrs_better']} of {binary['mboostrs_vs_reference']['n']}."
))

On regression, mboostrs has lower loss than the ridge reference on 27 of 33 tasks (Wilcoxon p = 3e-04) and higher loss than CatBoost on 24 of 33 (median loss ratio 1.28, p = 0.004). On binary tasks it beats the logistic reference on 23 of 30.

On the planted tasks the result depends on the structure. mboostrs has the
lowest loss of the six libraries on every additive task. On the pure interaction
S2 its loss is almost equal to that of the linear reference (9.324 against
9.334), because `fit` adopts no interaction term.

In [5]:
planted = analysis["D11"]["planted_by_task"]
labels = {"S1": "S1 smooth additive", "S2": "S2 pure interaction", "S3": "S3 spread only",
          "S4": "S4 no effect", "S5": "S5 200 rows", "S6": "S6 near-duplicates", "S7": "S7 binary additive"}
display(table(
    pl.DataFrame([{"task": labels[t], **{NAMES[a]: planted[t][a] for a in ARMS}} for t in planted]),
    stub="task", stubhead="Planted task",
    title="Test loss on the planted tasks",
    subtitle="Mean over five draws; MSE, or log loss for S7; lower is better",
    better={NAMES[a]: "min" for a in ARMS},
))

GT(_tbl_data=shape: (7, 7)
┌─────────────────────┬──────────┬──────────┬──────────┬─────────┬────────┬────────────────┐
│ task                ┆ mboostrs ┆ CatBoost ┆ LightGBM ┆ XGBoost ┆ EBM    ┆ Ridge/logistic │
│ ---                 ┆ ---      ┆ ---      ┆ ---      ┆ ---     ┆ ---    ┆ ---            │
│ str                 ┆ f64      ┆ f64      ┆ f64      ┆ f64     ┆ f64    ┆ f64            │
╞═════════════════════╪══════════╪══════════╪══════════╪═════════╪════════╪════════════════╡
│ S1 smooth additive  ┆ 0.2491   ┆ 0.2788   ┆ 0.3013   ┆ 0.34    ┆ 0.2576 ┆ 0.7053         │
│ S2 pure interaction ┆ 9.3236   ┆ 0.428    ┆ 0.9184   ┆ 1.7118  ┆ 0.7385 ┆ 9.3338         │
│ S3 spread only      ┆ 1.1842   ┆ 1.3197   ┆ 1.349    ┆ 1.472   ┆ 1.2259 ┆ 1.3514         │
│ S4 no effect        ┆ 1.0088   ┆ 1.1101   ┆ 1.1331   ┆ 1.2393  ┆ 1.0099 ┆ 1.0149         │
│ S5 200 rows         ┆ 0.2652   ┆ 0.4049   ┆ 0.3963   ┆ 0.5766  ┆ 0.3381 ┆ 0.7401         │
│ S6 near-duplicates  ┆ 0.2511   ┆ 0.2879   ┆ 0.3      ┆ 0.3431  ┆ 0.2636 ┆ 0.7209         │
│ S7 binary additive  ┆ 0.5266   ┆ 0.5491   ┆ 0.6086   ┆ 0.7238  ┆ 0.5323 ┆ 0.5311         │
└─────────────────────┴──────────┴──────────┴──────────┴─────────┴────────┴────────────────┘, _body=<great_tables._gt_data.Body object at 0x10caf7350>, _boxhead=Boxhead([ColInfo(var='task', type=<ColInfoTypeEnum.stub: 2>, column_label='task', column_align='left', column_width=None, stub_level=0), ColInfo(var='mboostrs', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs', column_align='right', column_width=None, stub_level=None), ColInfo(var='CatBoost', type=<ColInfoTypeEnum.default: 1>, column_label='CatBoost', column_align='right', column_width=None, stub_level=None), ColInfo(var='LightGBM', type=<ColInfoTypeEnum.default: 1>, column_label='LightGBM', column_align='right', column_width=None, stub_level=None), ColInfo(var='XGBoost', type=<ColInfoTypeEnum.default: 1>, column_label='XGBoost', column_align='right', column_width=None, stub_level=None), ColInfo(var='EBM', type=<ColInfoTypeEnum.default: 1>, column_label='EBM', column_align='right', column_width=None, stub_level=None), ColInfo(var='Ridge/logistic', type=<ColInfoTypeEnum.default: 1>, column_label='Ridge/logistic', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10cfacfe0>, _spanners=Spanners([]), _heading=Heading(title='Test loss on the planted tasks', subtitle='Mean over five draws; MSE, or log loss for S7; lower is better', preheader=None), _stubhead='Planted task', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d04af00>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d04b920>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10d048bc0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10d04bfe0>, <great_tables._gt_data.FormatInfo object at 0x10d09c0e0>, <great_tables._gt_data.FormatInfo object at 0x10d09c2f0>, <great_tables._gt_data.FormatInfo object at 0x10d09c1a0>, <great_tables._gt_data.FormatInfo object at 0x10d09c5c0>, <great_tables._gt_data.FormatInfo object at 0x10d09c530>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10d09c6e0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=Opti

## Discovery from data alone (D1b)

The same rule decides, for every library, which columns its fitted model uses: a
column counts as found when randomly permuting its values on the test rows
raises the test loss by more than twice the standard error of that increase and
by more than 0.5%.
The true-positive rate counts planted predictors found; the false-discovery
rate is the share of found columns that have no effect. mboostrs also states a
formula, and the last column scores that formula as the library's own claim.

In [6]:
d1b = analysis["D1b"]
d1b_cols = ARMS + ["mboostrs_native"]
d1b_names = {**NAMES, "mboostrs_native": "mboostrs formula"}
rows = [
    {"measure": "Linear effects found", **{d1b_names[a]: d1b[a]["tpr_by_type"]["linear"] for a in d1b_cols}},
    {"measure": "Nonlinear effects found", **{d1b_names[a]: d1b[a]["tpr_by_type"]["nonlinear"] for a in d1b_cols}},
    {"measure": "Interaction effects found", **{d1b_names[a]: d1b[a]["tpr_by_type"]["interaction"] for a in d1b_cols}},
    {"measure": "False-discovery rate (%)", **{d1b_names[a]: f"{100 * d1b[a]['fdr']:.1f}" for a in d1b_cols}},
    {"measure": "No-effect tasks with no finding", **{d1b_names[a]: d1b[a]["null_task_silent"] for a in d1b_cols}},
]
display(table(
    pl.DataFrame(rows), stub="measure", stubhead="Measure",
    title="Planted predictors found and false findings",
    subtitle="Planted tasks, five draws each; counts are found / planted",
    notes=("A near-duplicate copy (S6) or a spread-only column (S3) is counted as neither true nor false.",),
))

GT(_tbl_data=shape: (5, 8)
┌──────────────────┬──────────┬──────────┬──────────┬─────────┬───────┬─────────────────┬──────────┐
│ measure          ┆ mboostrs ┆ CatBoost ┆ LightGBM ┆ XGBoost ┆ EBM   ┆ Ridge/logistic  ┆ mboostrs │
│ ---              ┆ ---      ┆ ---      ┆ ---      ┆ ---     ┆ ---   ┆ ---             ┆ formula  │
│ str              ┆ str      ┆ str      ┆ str      ┆ str     ┆ str   ┆ str             ┆ ---      │
│                  ┆          ┆          ┆          ┆         ┆       ┆                 ┆ str      │
╞══════════════════╪══════════╪══════════╪══════════╪═════════╪═══════╪═════════════════╪══════════╡
│ Linear effects   ┆ 30/30    ┆ 30/30    ┆ 30/30    ┆ 30/30   ┆ 30/30 ┆ 30/30           ┆ 30/30    │
│ found            ┆          ┆          ┆          ┆         ┆       ┆                 ┆          │
│ Nonlinear        ┆ 40/40    ┆ 40/40    ┆ 40/40    ┆ 40/40   ┆ 40/40 ┆ 27/40           ┆ 40/40    │
│ effects found    ┆          ┆          ┆          ┆         ┆       ┆                 ┆          │
│ Interaction      ┆ 0/10     ┆ 10/10    ┆ 10/10    ┆ 10/10   ┆ 10/10 ┆ 0/10            ┆ 7/10     │
│ effects found    ┆          ┆          ┆          ┆         ┆       ┆                 ┆          │
│ False-discovery  ┆ 1.4      ┆ 41.2     ┆ 31.0     ┆ 45.9    ┆ 31.0  ┆ 3.4             ┆ 64.5     │
│ rate (%)         ┆          ┆          ┆          ┆         ┆       ┆                 ┆          │
│ No-effect tasks  ┆ 5/5      ┆ 1/5      ┆ 1/5      ┆ 1/5     ┆ 5/5   ┆ 5/5             ┆ 0/5      │
│ with no findin…  ┆          ┆          ┆          ┆         ┆       ┆                 ┆          │
└──────────────────┴──────────┴──────────┴──────────┴─────────┴───────┴─────────────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10b214e30>, _boxhead=Boxhead([ColInfo(var='measure', type=<ColInfoTypeEnum.stub: 2>, column_label='measure', column_align='left', column_width=None, stub_level=0), ColInfo(var='mboostrs', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs', column_align='right', column_width=None, stub_level=None), ColInfo(var='CatBoost', type=<ColInfoTypeEnum.default: 1>, column_label='CatBoost', column_align='right', column_width=None, stub_level=None), ColInfo(var='LightGBM', type=<ColInfoTypeEnum.default: 1>, column_label='LightGBM', column_align='right', column_width=None, stub_level=None), ColInfo(var='XGBoost', type=<ColInfoTypeEnum.default: 1>, column_label='XGBoost', column_align='right', column_width=None, stub_level=None), ColInfo(var='EBM', type=<ColInfoTypeEnum.default: 1>, column_label='EBM', column_align='right', column_width=None, stub_level=None), ColInfo(var='Ridge/logistic', type=<ColInfoTypeEnum.default: 1>, column_label='Ridge/logistic', column_align='right', column_width=None, stub_level=None), ColInfo(var='mboostrs formula', type=<ColInfoTypeEnum.default: 1>, column_label='mboostrs formula', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10cfacfe0>, _spanners=Spanners([]), _heading=Heading(title='Planted predictors found and false findings', subtitle='Planted tasks, five draws each; counts are found / planted', preheader=None), _stubhead='Measure', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d09cd40>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d09c170>, _source_notes=['A near-duplicate copy (S6) or a spread-only column (S3) is counted as neither true nor false.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10d09c3b0>, _formats=[], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10d09c7d0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True,

The tree libraries and EBM find every planted predictor, including the
interaction, and also report between 31% and 46% false columns. The model mboostrs
fits finds no interaction and reports almost no false structure. Its stated
formula keeps the noise columns that the fitted model gives almost no weight, so
it has the highest false-discovery rate in the table: what mboostrs states is less
reliable than what it fits.

On the real tables, a second check asks whether the direction of each fitted
effect agrees with the published analysis of the dataset. For 38 effect
directions published for 18 well-studied datasets, the column was moved from
its 10th to its 90th percentile and the sign of the change in prediction was
compared with the published sign. On the
75 (effect, seed) pairs that every library completed, every library recovered
every direction, so this check does not separate the libraries.

## Uncertainty (D9)

Each library's own 90% prediction interval on the regression tasks: split
conformal for mboostrs (`mb.conformal_interval` with `method="split"`: the
model is refitted on part of the training rows and the band is set from the
residuals of the remaining rows, so that it covers new observations at the
stated rate), quantile regression for the tree libraries and OLS (ordinary least squares) intervals for
the reference. EBM has no interval for a new observation.

In [7]:
d9 = analysis["D9"]
rows = [{"library": NAMES[a],
         "median_coverage": d9[a]["median_coverage"],
         "share_in_band": None if d9[a]["share_within_085_095"] is None else 100 * d9[a]["share_within_085_095"],
         "s3_low": None if d9[a]["s3_coverage_by_x2_tercile"] is None else d9[a]["s3_coverage_by_x2_tercile"][0],
         "s3_mid": None if d9[a]["s3_coverage_by_x2_tercile"] is None else d9[a]["s3_coverage_by_x2_tercile"][1],
         "s3_high": None if d9[a]["s3_coverage_by_x2_tercile"] is None else d9[a]["s3_coverage_by_x2_tercile"][2]}
        for a in ARMS]
display(table(
    pl.DataFrame(rows, schema_overrides={k: pl.Float64 for k in rows[0] if k != "library"}),
    stub="library", stubhead="Library",
    title="Coverage of the 90% prediction interval",
    subtitle="Real and planted regression runs; target coverage 0.90",
    labels={"median_coverage": "Median coverage", "share_in_band": "Runs with 0.85–0.95 (%)",
            "s3_low": "Low", "s3_mid": "Middle", "s3_high": "High"},
    spanners={"All runs": ["median_coverage", "share_in_band"],
              "S3 coverage by tercile of the spread column": ["s3_low", "s3_mid", "s3_high"]},
    formats={"share_in_band": 0},
    notes=("EBM offers no prediction interval for a new observation.",),
))

GT(_tbl_data=shape: (6, 6)
┌────────────────┬─────────────────┬───────────────┬────────┬────────┬─────────┐
│ library        ┆ median_coverage ┆ share_in_band ┆ s3_low ┆ s3_mid ┆ s3_high │
│ ---            ┆ ---             ┆ ---           ┆ ---    ┆ ---    ┆ ---     │
│ str            ┆ f64             ┆ f64           ┆ f64    ┆ f64    ┆ f64     │
╞════════════════╪═════════════════╪═══════════════╪════════╪════════╪═════════╡
│ mboostrs       ┆ 0.9             ┆ 96.3          ┆ 0.892  ┆ 0.906  ┆ 0.908   │
│ CatBoost       ┆ 0.752           ┆ 9.1           ┆ 0.661  ┆ 0.726  ┆ 0.716   │
│ LightGBM       ┆ 0.79            ┆ 7.9           ┆ 0.811  ┆ 0.763  ┆ 0.738   │
│ XGBoost        ┆ 0.733           ┆ 7.9           ┆ 0.646  ┆ 0.686  ┆ 0.666   │
│ EBM            ┆ null            ┆ null          ┆ null   ┆ null   ┆ null    │
│ Ridge/logistic ┆ 0.914           ┆ 73.3          ┆ 0.995  ┆ 0.915  ┆ 0.787   │
└────────────────┴─────────────────┴───────────────┴────────┴────────┴─────────┘, _body=<great_tables._gt_data.Body object at 0x10d04af00>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None, stub_level=0), ColInfo(var='median_coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Median coverage', column_align='right', column_width=None, stub_level=None), ColInfo(var='share_in_band', type=<ColInfoTypeEnum.default: 1>, column_label='Runs with 0.85–0.95 (%)', column_align='right', column_width=None, stub_level=None), ColInfo(var='s3_low', type=<ColInfoTypeEnum.default: 1>, column_label='Low', column_align='right', column_width=None, stub_level=None), ColInfo(var='s3_mid', type=<ColInfoTypeEnum.default: 1>, column_label='Middle', column_align='right', column_width=None, stub_level=None), ColInfo(var='s3_high', type=<ColInfoTypeEnum.default: 1>, column_label='High', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10d09c650>, _spanners=Spanners([SpannerInfo(spanner_id='All runs', spanner_level=0, spanner_label='All runs', spanner_units=None, spanner_pattern=None, vars=['median_coverage', 'share_in_band'], built=None), SpannerInfo(spanner_id='S3 coverage by tercile of the spread column', spanner_level=0, spanner_label='S3 coverage by tercile of the spread column', spanner_units=None, spanner_pattern=None, vars=['s3_low', 's3_mid', 's3_high'], built=None)]), _heading=Heading(title='Coverage of the 90% prediction interval', subtitle='Real and planted regression runs; target coverage 0.90', preheader=None), _stubhead='Library', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d09cc50>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d09d040>, _source_notes=['EBM offers no prediction interval for a new observation.'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09D3D0>, mask=None), grpname=None, colname='median_coverage', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09D3D0>, mask=None), grpname=None, colname='share_in_band', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09D3D0>, mask=None), grpname=None, colname='s3_low', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09D3D0>, mask=None), grpname=None, colname='s3_mid', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09D3D0>, mask=None), grpname=None, colname='s3_high', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')])], _locale=<great_tables._gt_data.Locale object at 0x10

The mboostrs interval covers 90% at the median and stays near 90% where the
noise is small and where it is large (S3). The quantile intervals of the tree
libraries cover 73% to 79%. The OLS interval of the reference has the right
average coverage and the wrong shape: it over-covers where the noise is small
and under-covers where it is large.

## Speed (D5) and robustness (D6)

The D5 measurement is the scorecard's analyst session: on a 50,000 × 20 table,
ten follow-up fits, each with prediction and permutation importance. The D6
measurement is the share of the 285 real-data runs that complete.

In [8]:
rows = [{"library": NAMES[a],
         "session_median": analysis["D5"]["session"][a]["median_seconds"],
         "session_p90": analysis["D5"]["session"][a]["p90_seconds"],
         "completed": analysis["D6"][a]["completed"],
         "s5_loss": analysis["D6"][a]["S5_loss"]}
        for a in ARMS]
display(table(
    pl.DataFrame(rows), stub="library", stubhead="Library",
    title="Follow-up fit time and completed runs",
    subtitle="Seconds per follow-up fit, one thread; runs completed of 285",
    labels={"session_median": "Median (s)", "session_p90": "90th percentile (s)",
            "completed": "Real runs completed", "s5_loss": "S5 loss (200 rows)"},
    spanners={"D5 session": ["session_median", "session_p90"], "D6": ["completed", "s5_loss"]},
    better={"session_median": "min", "s5_loss": "min"},
    formats={"session_p90": 2},
    decimals=3,
    notes=("EBM's 25 failures are the 900 s time limit on wide tables.",),
))

GT(_tbl_data=shape: (6, 5)
┌────────────────┬────────────────┬─────────────┬───────────┬─────────┐
│ library        ┆ session_median ┆ session_p90 ┆ completed ┆ s5_loss │
│ ---            ┆ ---            ┆ ---         ┆ ---       ┆ ---     │
│ str            ┆ f64            ┆ f64         ┆ i64       ┆ f64     │
╞════════════════╪════════════════╪═════════════╪═══════════╪═════════╡
│ mboostrs       ┆ 34.395         ┆ 59.872      ┆ 202       ┆ 0.2652  │
│ CatBoost       ┆ 6.685          ┆ 8.306       ┆ 285       ┆ 0.4049  │
│ LightGBM       ┆ 0.76           ┆ 1.202       ┆ 285       ┆ 0.3963  │
│ XGBoost        ┆ 0.58           ┆ 0.859       ┆ 285       ┆ 0.5766  │
│ EBM            ┆ 67.515         ┆ 90.3        ┆ 260       ┆ 0.3381  │
│ Ridge/logistic ┆ 0.055          ┆ 0.15        ┆ 285       ┆ 0.7401  │
└────────────────┴────────────────┴─────────────┴───────────┴─────────┘, _body=<great_tables._gt_data.Body object at 0x10d09c9b0>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None, stub_level=0), ColInfo(var='session_median', type=<ColInfoTypeEnum.default: 1>, column_label='Median (s)', column_align='right', column_width=None, stub_level=None), ColInfo(var='session_p90', type=<ColInfoTypeEnum.default: 1>, column_label='90th percentile (s)', column_align='right', column_width=None, stub_level=None), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='Real runs completed', column_align='right', column_width=None, stub_level=None), ColInfo(var='s5_loss', type=<ColInfoTypeEnum.default: 1>, column_label='S5 loss (200 rows)', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x10cfacfe0>, _spanners=Spanners([SpannerInfo(spanner_id='D5 session', spanner_level=0, spanner_label='D5 session', spanner_units=None, spanner_pattern=None, vars=['session_median', 'session_p90'], built=None), SpannerInfo(spanner_id='D6', spanner_level=0, spanner_label='D6', spanner_units=None, spanner_pattern=None, vars=['completed', 's5_loss'], built=None)]), _heading=Heading(title='Follow-up fit time and completed runs', subtitle='Seconds per follow-up fit, one thread; runs completed of 285', preheader=None), _stubhead='Library', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d09cf80>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d09c620>, _source_notes=["EBM's 25 failures are the 900 s time limit on wide tables."], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09D400>, mask=None), grpname=None, colname='session_median', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09D400>, mask=None), grpname=None, colname='session_p90', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09D400>, mask=None), grpname=None, colname='completed', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x10D09D400>, mask=None), grpname=None, colname='s5_loss', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')])], _locale=<great_tables._gt_data.Locale object at 0x10d09c530>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10bd076b0>, <great_tables._gt_data.FormatInfo object at 0x10d09c0e0>, <great_tables._gt_data.FormatInfo object at 0x10d09d2e0>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x10d09d010>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, categor

mboostrs fails under three input conditions: a factor level at prediction that
the training rows do not contain, a missing value at prediction, and tables in
which every row has a missing value in some column. With 200 training rows
(S5), mboostrs has the lowest loss.

## Reproducibility (D10)

Ten repeated runs of the same task and seed gave identical predictions for
every library. mboostrs is the only library that records how each model was
chosen: `fit` returns a decision trail with the stages of the search, the fits
it used and the reason each candidate was kept or dropped.

## `fit` against its previous version

The previous version of `fit` ran the specification search with no budget,
which in practice gave every column its default learner. Both versions were
run on the same 95 tables with seeds 4000, 4001 and 4002, which were not used
in development. A table enters the paired comparison when both versions
complete it on all three seeds; its loss ratio is the geometric mean of the
three per-seed ratios.

In [9]:
v4 = json.loads((ROOT / "docs/benchmarks/fit_heldout/analysis_v4.json").read_text())["new"]
a, b, c = v4["A_all"], v4["B_tier"], v4["C_known_effects"]
sec = v4["secondary"]
display(Markdown(
    f"- **All datasets.** Held-out loss is lower on {a['better']} of {a['n']} paired datasets and "
    f"higher on {a['worse']} (Wilcoxon p = {a['wilcoxon_p']:.0e}); the median loss ratio is "
    f"{a['median_ratio']:.3f}. The current version fails {a['failures_arm']} runs, the previous "
    f"one {a['failures_old']}.\n"
    f"- **Well-studied datasets.** {b['better']} better and {b['worse']} worse "
    f"(p = {b['wilcoxon_p']:.2f}): no significant gain.\n"
    f"- **Published effect directions.** {c['recovered_arm']} of {c['pairs']} recovered "
    f"({100 * c['rate_arm']:.1f}%) against {c['recovered_old']} ({100 * c['rate_old']:.1f}%).\n"
    f"- **Time.** A median {sec['median_seconds'][1]:.1f} s per run against "
    f"{sec['median_seconds'][0]:.2f} s."
))

- **All datasets.** Held-out loss is lower on 38 of 62 paired datasets and higher on 20 (Wilcoxon p = 8e-05); the median loss ratio is 0.990. The current version fails 79 runs, the previous one 88.
- **Well-studied datasets.** 8 better and 5 worse (p = 0.24): no significant gain.
- **Published effect directions.** 76 of 77 recovered (98.7%) against 73 (94.8%).
- **Time.** A median 5.9 s per run against 0.11 s.

The median loss ratio of 0.990 means that on half of the 62 tables the loss
changes by about 1% or less. On 28 tables the loss falls by more than 5%, and
on 12 by more than 80%. The largest reductions are on tables with exact or
near-exact structure, such as `synchronous_machine`, `wave_energy` and
`energy_efficiency`, where the search adopts a learner that the default
specification lacks.

## Open problems

1. **Completion on real data (D6).** The three failure conditions above end 29%
   of the real-data runs. They also remove 12 regression and 20 binary tasks
   from the prediction comparison of mboostrs.
2. **Interaction capacity (D11, D1b).** The planted interaction S2 and the
   real-data gap to CatBoost both indicate that the search does not adopt
   interaction terms the data support.
3. **The stated formula (D1b).** The formula that mboostrs reports keeps noise
   columns to which the fitted model gives almost no weight, so its
   false-discovery rate is 64.5% against 1.4% for the fitted model.
4. **Speed of follow-up fits (D5).** 34 seconds per follow-up fit at 50,000
   rows.

## Limitations

- No library was tuned. Tuning changes each library's results by a different
  amount; [](competitive_shootout.ipynb) reports a study with tuning grids.
- The 95 real tables were used in the development of `fit`, in the comparison
  with its previous version; the other libraries were not adjusted to them.
  This favours mboostrs.
- The check of published effects on real data uses only the sign of a change
  from the 10th to the 90th percentile.
- All runs used one machine (Apple M4) with eight concurrent single-thread runs.
- D1a, D2, D3, D4, D7 and D8 are not measured. D8 requires an agent study.

## Data and code

The protocols, deviations, per-run records, analysis and one scorecard report
per library are in `docs/benchmarks/fit_heldout/` and
`docs/benchmarks/competitive_heldout/`; the runners are in
`benches/fit_heldout/` and `benches/competitive_heldout/`.